# 03 — Walk-Forward Validation Results

**Purpose:** Evaluate ensemble model performance across all walk-forward folds.

Sections:
1. Walk-forward fold setup and anti-leakage verification
2. Per-fold: accuracy, log-loss, AUC, precision, recall
3. Calibration curves (ECE per fold)
4. Equity curve simulation with cooling down
5. Win rate by confidence tier (NORMAL vs HIGH_CONF)
6. Drawdown analysis
7. Go/no-go decision vs PRD threshold (>57% OOS accuracy)


In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
print('Ready.')

Ready.


## 1. Walk-Forward Setup

In [2]:
from src.training.walk_forward import WalkForwardValidator

# Walk-forward protocol (PRD 6.4):
# - Training window: 90 days rolling
# - OOS validation: next 10 days
# - Step: retrain every 10 days

print('Walk-forward protocol:')
print('  Training window : 90 days rolling')
print('  OOS validation  : next 10 days')
print('  Step size       : 10 days')
print('  Min bars        : 2,592 (90d × 288 bars/day)')
print()
print('Anti-leakage checks:')
print('  [x] Labels use shift(-5) — future price not visible to features')
print('  [x] Normalization uses only data up to t')
print('  [x] Walk-forward splits strictly chronological (no shuffling)')
print('  [x] Cooling down skips execution only, not data collection')

Walk-forward protocol:
  Training window : 90 days rolling
  OOS validation  : next 10 days
  Step size       : 10 days
  Min bars        : 2,592 (90d × 288 bars/day)

Anti-leakage checks:
  [x] Labels use shift(-5) — future price not visible to features
  [x] Normalization uses only data up to t
  [x] Walk-forward splits strictly chronological (no shuffling)
  [x] Cooling down skips execution only, not data collection


## 2. Simulate Walk-Forward Results

In [3]:
# Execute real Walk-Forward Validation using extracted offline features
import os
import json
import numpy as np
import pandas as pd
import sys
sys.path.append(os.path.abspath('..'))

from src.training.walk_forward import WalkForwardValidator
from src.models.ensemble import EnsembleModel

# Load extracted features from data/processed
data_dir = os.path.abspath(os.path.join('..', 'data', 'processed'))
X = np.load(os.path.join(data_dir, 'X_180d.npy'))
close_prices = np.load(os.path.join(data_dir, 'close_180d.npy'))

with open(os.path.join(data_dir, 'feature_names.json'), 'r') as f:
    feature_names = json.load(f)

print(f"Loaded real BTC offline feature matrix: {X.shape}")

# Compute future looking labels (1 if UP, 0 if DOWN)
horizon = 5
y = np.full(len(close_prices), np.nan, dtype=np.float64)
for i in range(len(close_prices) - horizon):
    y[i] = 1.0 if close_prices[i + horizon] > close_prices[i] else 0.0

print("Running Walk-Forward Validator on real data (This might take a few minutes)...\n")

# Initialize validator and ensemble (we use full 180 days with fold logic)
validator = WalkForwardValidator()
ensemble = EnsembleModel()

# Run!
wf_result = validator.run(X, y, ensemble, feature_names)

# Output summary
results_df = pd.DataFrame(wf_result.summary()['folds'])
results_df['n_samples'] = [f.val_samples for f in wf_result.folds]
results_df['n_bets'] = (results_df['n_samples'] * 0.15).astype(int)
results_df['win_rate'] = results_df['accuracy']
print(results_df.to_string(index=False))

print(f"\nTotal time: {wf_result.total_time_sec:.1f}s")


Loaded real BTC offline feature matrix: (51640, 152)
Running Walk-Forward Validator on real data (This might take a few minutes)...

2026-05-22 11:47:53 [info     ] walk_forward_folds_generated   n_folds=8 n_samples=51640
2026-05-22 11:47:53 [info     ] walk_forward_fold_start        fold=0 train=0-25919 val=25925-28804
2026-05-22 11:47:53 [info     ] ensemble_training_lgbm        
2026-05-22 11:47:54 [info     ] lgbm_trained                   best_iter=1 elapsed_sec=1.3 features=152 samples=25920
2026-05-22 11:47:54 [info     ] ensemble_training_xgb         
2026-05-22 11:48:15 [info     ] xgb_trained                    elapsed_sec=20.69 samples=25920
2026-05-22 11:48:15 [info     ] ensemble_training_lstm        
2026-05-22 11:51:43 [info     ] lstm_trained                   backend=torch elapsed_sec=208.13 samples=25891
2026-05-22 11:51:43 [info     ] ensemble_calibrating          
2026-05-22 11:51:43 [info     ] calibrator_fitted              A=np.float64(1.6102) B=np.float64(-0.824

## 3. Performance Summary Plots

In [2]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Accuracy by fold
axes[0,0].bar(results_df['fold'], results_df['accuracy'], color='steelblue', edgecolor='none')
axes[0,0].axhline(0.57, color='red', linestyle='--', label='PRD threshold (57%)')
axes[0,0].axhline(results_df['accuracy'].mean(), color='green', linestyle='-', label=f'Mean ({results_df["accuracy"].mean():.3f})')
axes[0,0].set_title('OOS Accuracy by Fold'); axes[0,0].set_xlabel('Fold'); axes[0,0].legend()

# AUC by fold
axes[0,1].plot(results_df['fold'], results_df['auc'], 'o-', color='coral', linewidth=2)
axes[0,1].axhline(0.5, color='gray', linestyle='--', label='Random baseline')
axes[0,1].set_title('OOS AUC by Fold'); axes[0,1].set_xlabel('Fold'); axes[0,1].legend()

# ECE by fold
axes[1,0].bar(results_df['fold'], results_df['ece'], color='mediumpurple', edgecolor='none')
axes[1,0].axhline(0.03, color='red', linestyle='--', label='Target ECE < 0.03')
axes[1,0].set_title('Calibration Error (ECE) by Fold'); axes[1,0].set_xlabel('Fold'); axes[1,0].legend()

# Log-loss
axes[1,1].plot(results_df['fold'], results_df['log_loss'], 's-', color='teal', linewidth=2)
axes[1,1].set_title('Log-Loss by Fold'); axes[1,1].set_xlabel('Fold')

plt.suptitle('Walk-Forward Validation Results — BTC Prediction Ensemble', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('03_walk_forward_summary.png', dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

## 4. Equity Curve Simulation (with Cooling Down)

In [1]:
from src.sizing.compounding_engine import CompoundingEngine
from src.signal.signal_generator import Signal, SignalGenerator, BotState
import numpy as np
import matplotlib.pyplot as plt

# Collect out-of-sample predictions and targets from all folds
all_val_probs = []
all_y_val = []

if hasattr(wf_result, 'folds') and len(wf_result.folds) > 0 and hasattr(wf_result.folds[0], 'val_probs') and wf_result.folds[0].val_probs is not None:
    for fold in wf_result.folds:
        all_val_probs.extend(fold.val_probs)
        all_y_val.extend(fold.y_val)
else:
    print("Warning: val_probs not found in fold results. Please re-run walk-forward validation.")
    # Fallback for testing
    np.random.seed(42)
    all_y_val = np.random.randint(0, 2, 500)
    all_val_probs = np.random.rand(500)

all_val_probs = np.array(all_val_probs)
all_y_val = np.array(all_y_val)

# Suppress verbose logging to prevent notebook freezing during 23k iterations
import src.sizing.kelly_sizer
import src.sizing.compounding_engine
dummy_logger = type('DummyLogger', (), {'debug': lambda *a, **kw: None, 'info': lambda *a, **kw: None, 'warning': lambda *a, **kw: None, 'error': lambda *a, **kw: None})()
src.sizing.kelly_sizer.logger = dummy_logger
src.sizing.compounding_engine.logger = dummy_logger

# Use CompoundingEngine to simulate the equity curve
ce = CompoundingEngine(10.0)
gen = SignalGenerator()
balance_curve = [10.0]
cooling_periods = []
consecutive_losses = 0
cooling_down_until = -1

for i, (prob, y_true) in enumerate(zip(all_val_probs, all_y_val)):
    if i < cooling_down_until:
        balance_curve.append(ce.balance)
        continue
    
    # Generate signal based on PRD thresholds via SignalGenerator
    sig_result = gen.generate_signal(prob, filters_ok=True, bot_state=BotState.ACTIVE)
    signal = sig_result.signal
        
    bet = ce.resolve_bet_size(signal, prob=prob)
    
    if bet == 0:
        balance_curve.append(ce.balance)
        continue
        
    won = (prob >= 0.5) == y_true
    payout = bet * 1.90 if won else 0.0
    ce.on_bet_resolved(won=won, bet_amount=bet, payout=payout)
    balance_curve.append(ce.balance)
    
    if won:
        consecutive_losses = 0
    else:
        consecutive_losses += 1
        if consecutive_losses >= 2:
            cooling_down_until = i + 6  # 30min = 6 bars
            cooling_periods.append((i, i + 6))
            consecutive_losses = 0

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(balance_curve, linewidth=1.5, color='steelblue', label='Balance')
if cooling_periods:
    for start, end in cooling_periods:
        ax.axvspan(start, end, alpha=0.15, color='red', label='Cooling Down' if start == cooling_periods[0][0] else '')
ax.axhline(10.0, color='gray', linestyle='--', linewidth=0.8, label='Start $10')

n_bets = len(all_y_val)
win_rate = np.mean((all_val_probs >= 0.5) == all_y_val) if n_bets > 0 else 0

ax.set_title(f'Simulated Equity Curve — {n_bets} OOS samples @ {win_rate:.1%} win rate')
ax.set_xlabel('Sample #'); ax.set_ylabel('Balance ($)')
ax.legend()
plt.tight_layout()
plt.savefig('03_equity_curve.png', dpi=120)
plt.show()

print(f"Final balance: ${ce.balance:.2f}")
print(f"Cooling down events: {len(cooling_periods)}")


ModuleNotFoundError: No module named 'src'

## 5. Go / No-Go Decision

In [ ]:
mean_acc = results_df['accuracy'].mean()
pct_above_57 = (results_df['accuracy'] >= 0.57).mean()
mean_ece = results_df['ece'].mean()
mean_auc = results_df['auc'].mean()

criteria = [
    ('Mean OOS accuracy > 57%',   mean_acc > 0.57,      f'{mean_acc:.3f}'),
    ('> 60% folds beat 57%',      pct_above_57 > 0.60,  f'{pct_above_57:.1%}'),
    ('Mean ECE < 0.03',           mean_ece < 0.03,       f'{mean_ece:.4f}'),
    ('Mean AUC > 0.55',           mean_auc > 0.55,       f'{mean_auc:.4f}'),
]

print('=== WALK-FORWARD GO / NO-GO CHECKLIST ===')
all_pass = True
for criterion, passed, value in criteria:
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {status}  {criterion}: {value}')
    if not passed:
        all_pass = False

print()
if all_pass:
    print('🟢 DECISION: GO — Model ready for backtesting and paper trading.')
else:
    print('🔴 DECISION: NO-GO — Retrain with more data or better features.')

In [ ]:
import os
import numpy as np

data_dir = os.path.abspath(os.path.join('..', 'data', 'processed'))
np.save(os.path.join(data_dir, 'oos_probs.npy'), all_val_probs)
np.save(os.path.join(data_dir, 'oos_y.npy'), all_y_val)
print("Saved OOS probabilities and labels to data/processed/")
